## 1) Configuración inicial e imports
**Objetivo:** preparar el entorno de trabajo del notebook.  
**Qué se hace:**
- Se importan librerías base (`pandas`, `numpy`, `matplotlib`, `seaborn`, `Path`).
- Se configura `sys.path` para poder importar módulos desde `src/`.
- Se define el tema visual de gráficos.

**Resultado esperado:** entorno listo para ejecutar el EDA de forma reproducible.


In [ ]:
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

root = Path.cwd().parent

src_path = root / 'src'
if src_path not in sys.path:
	sys.path.insert(0, str(src_path))

sns.set_theme(context='notebook', style='whitegrid', palette='husl')
plt.rcParams['figure.figsize'] = (12, 6)

print(f"Imports completados")
print(f"Root: {root}")

## 2) Carga de módulos del proyecto (`src/`)
**Objetivo:** centralizar lógica en funciones reutilizables.  
**Qué se hace:**
- Se importan utilidades de I/O, limpieza, features, visualización y control de calidad.
- Se importan rutas y configuración desde `config.py`.

**Resultado esperado:** funciones internas disponibles para pipeline completo.


In [ ]:
from io_utils import load_csv, save_csv
from cleaning import clean_pipeline
from features import features_pipeline
from viz import *
from utils import qc_missing, qc_duplicates
from config import RAW_PATH, OUT_PATH, BASE_KEEP_COLS, get_keep_cols

print("Módulos src/ importados correctamente")

## 3) Carga del dataset raw
**Objetivo:** leer el dataset original desde la ruta de entrada.  
**Qué se hace:**
- Se carga el CSV usando `load_csv(RAW_PATH)`.
- Se inspeccionan dimensiones y primeras filas.

**Resultado esperado:** `df_raw` inicial validado visualmente.


In [ ]:
raw_path = RAW_PATH
df_raw = load_csv(raw_path)

print(f"\n{'='*60}")
print(f"DATASET CARGADO")
print(f"{'='*60}")
print(f"Forma: {df_raw.shape[0]} filas × {df_raw.shape[1]} columnas")
print(f"\nPrimeras filas:")
display(df_raw.head())

## 4) Selección de columnas relevantes
**Objetivo:** reducir el dataset a variables analíticas necesarias.  
**Qué se hace:**
- Se calcula `KEEP_COLS` dinámicamente con `get_keep_cols`.
- Se alertan columnas esperadas ausentes.
- Se eliminan columnas no incluidas en el set objetivo.

**Impacto del cambio:** menor ruido, menor coste computacional y foco analítico.


In [ ]:
KEEP_COLS = get_keep_cols(df_raw.columns)

missing = sorted(BASE_KEEP_COLS - set(df_raw.columns))
if missing:
	print(f"Faltan columnas esperadas: {missing}")

original_cols = set(df_raw.columns)
df_raw = df_raw[[c for c in KEEP_COLS if c in df_raw.columns]].copy()

dropped = sorted(original_cols - set(df_raw.columns))
print(f"Columnas conservadas: {len(df_raw.columns)}")
print(f"Columnas eliminadas: {len(dropped)}")
print(f"Ejemplo eliminadas: {dropped[:15]}")

## 5) Control de calidad inicial (QC)
**Objetivo:** diagnosticar estado del dato antes de limpiar.  
**Qué se hace:**
- Revisión de tipos (`dtypes`).
- Medición de valores nulos (`qc_missing`).
- Detección de duplicados (`qc_duplicates`).

**Resultado esperado:** línea base de calidad para comparar después de la limpieza.


In [ ]:
print(f"\n{'='*60}")
print(f"QC INICIAL (raw data)")
print(f"{'='*60}")

print(f"\nTipos de datos:")
print(df_raw.dtypes)

qc_missing(df_raw, verbose=True)
qc_duplicates(df_raw, verbose=True)

## 6) Limpieza de datos (`clean_pipeline`)
**Objetivo:** estandarizar y depurar datos para análisis fiable.  
**Qué se hace:**
- Se aplica `clean_pipeline(df_raw)`.

**Impacto del cambio:** `df_clean` queda listo para feature engineering.


In [ ]:
df_clean = clean_pipeline(df_raw)

## 7) Features (`features_pipeline`)
**Objetivo:** crear nuevas variables explicativas de negocio.  
**Qué se hace:**
- Se generan variables derivadas (`price_per_person`, `amenities_count`).

**Impacto del cambio:** mayor capacidad explicativa para responder preguntas analíticas.


In [ ]:
df_feat = features_pipeline(df_clean)

print(f"Nuevas columnas creadas:")
print(f"- price_per_person")
print(f"- amenities_count")

## 8) Persistencia del dataset procesado
**Objetivo:** guardar salida intermedia/final para reutilización.  
**Qué se hace:**
- Se exporta `df_feat` a `OUT_PATH` con `save_csv`.

**Resultado esperado:** dataset procesado disponible para modelado/reporting.


In [ ]:
processed_path = OUT_PATH
save_csv(df_feat, processed_path)

print(f"✓ Dataset procesado guardado en: {processed_path}")

## 9) Estadística descriptiva univariante
**Objetivo:** entender distribución básica de variables clave.  
**Qué se hace:**
- Resumen estadístico de `price`, `accommodates`, `number_of_reviews`, `availability_365`.

**Valor analítico:** detección temprana de sesgos, dispersión y posibles extremos.


In [ ]:
print(f"\n{'='*60}")
print(f"ESTADÍSTICAS DESCRIPTIVAS (después de limpiar)")
print(f"{'='*60}")

print(f"\n--- PRECIO ---")
print(df_feat['price'].describe())

print(f"\n--- ACOMODACIÓN ---")
print(df_feat['accommodates'].describe())

print(f"\n--- RESEÑAS ---")
print(df_feat['number_of_reviews'].describe())

print(f"\n--- DISPONIBILIDAD ANUAL ---")
print(df_feat['availability_365'].describe())

## 10) Análisis de variables categóricas
**Objetivo:** caracterizar composición del mercado por categorías.  
**Qué se hace:**
- Conteo de valores únicos y frecuencias para tipo de alojamiento, propiedad, barrio y superhost.

**Valor analítico:** identificación de categorías dominantes.


In [ ]:
print(f"\n{'='*60}")
print(f"VARIABLES CATEGÓRICAS")
print(f"{'='*60}")

cat_cols = ['room_type', 'property_type', 'neighbourhood_cleansed', 'host_is_superhost']

for col in cat_cols:
	if col in df_feat.columns:
		print(f"\n--- {col.upper()} ---")
		print(f"Valores únicos: {df_feat[col].nunique()}")
		print(f"\nValue counts:")
		print(df_feat[col].value_counts().head(10))

## 11) Pregunta 1 — Factores asociados al precio
**Objetivo:** identificar variables numéricas relacionadas con `price`.  
**Qué se hace:**
- Matriz de correlación y ranking de correlaciones con precio.
- Heatmap de apoyo visual.

**Lectura esperada:** priorización de drivers cuantitativos del precio.


In [ ]:
print(f"\n{'='*80}")
print(f"PREGUNTA 1: ¿Qué factores explican mejor el precio de un alojamiento?")
print(f"{'='*80}")

viz_path = root / 'visualizations'
viz_path.mkdir(parents=True, exist_ok=True)

# ── Figura 1: Heatmap de correlación ──
corr_with_price = plot_correlation_heatmap(
	df_feat, save_path=viz_path / 'p1_heatmap_correlacion.png'
)

# ── Imprimir correlaciones ──
print(f"\nCORRELACIONES con precio (Pearson):")
for var, val in corr_with_price.items():
	signo = "🟢" if val > 0.3 else ("🟡" if val > 0.1 else "🔴")
	print(f"  {signo} {var:<25s} {val:+.3f}")

# ── Figura 2: Ranking de correlación ──
plot_correlation_ranking(
	corr_with_price, save_path=viz_path / 'p1_ranking_correlacion_precio.png'
)

# ── Insight resumen ──
top_var = corr_with_price.index[0]
top_val = corr_with_price.values[0]
print(f"\n{'─'*80}")
print(f"📊 INSIGHT — Pregunta 1: Factores del precio")
print(f"{'─'*80}")
print(f"  • Driver principal: {top_var.upper()} (r = {top_val:+.3f})")
print(f"  • Variables de capacidad (accommodates, bedrooms, beds) dominan el ranking.")
print(f"  • amenities_count tiene efecto positivo moderado.")
print(f"  • Las reseñas y ratings tienen relación débil/negativa con el precio.")
print(f"  → Los hosts deben priorizar CAPACIDAD + COMODIDADES para maximizar tarifa.")
print(f"{'─'*80}")

## 12) Pregunta 2 — Diferencias de precio por tipo y barrio
**Objetivo:** comparar precio entre segmentos geográficos y de producto.  
**Qué se hace:**
- Agregados por `room_type`.
- Agregados por barrios más frecuentes.
- Visualizaciones comparativas.

**Lectura esperada:** evidencia de heterogeneidad de precios por ubicación y tipo.


In [ ]:
print(f"\n{'='*80}")
print(f"PREGUNTA 2: ¿Diferencias significativas de precio entre barrios y tipos?")
print(f"{'='*80}")

viz_path = root / 'visualizations'

# ══════════════════════════════════════════════════════════════════════
# PARTE A: Estadísticas por tipo de alojamiento
# ══════════════════════════════════════════════════════════════════════
print(f"\n── Precio por TIPO DE ALOJAMIENTO ──")
rt_stats = (
	df_feat.groupby('room_type')['price']
	.agg(['count', 'mean', 'median', 'std'])
	.sort_values('median', ascending=False)
	.round(2)
)
rt_stats['share_%'] = (rt_stats['count'] / rt_stats['count'].sum() * 100).round(1)
print(rt_stats)

# ══════════════════════════════════════════════════════════════════════
# PARTE B: Estadísticas por barrio (Top 15)
# ══════════════════════════════════════════════════════════════════════
print(f"\n── Precio por BARRIO (Top 15 por volumen) ──")
top_n = 15
top_barrios = df_feat['neighbourhood_cleansed'].value_counts().head(top_n).index
df_top = df_feat[df_feat['neighbourhood_cleansed'].isin(top_barrios)]

barrio_stats = (
	df_top.groupby('neighbourhood_cleansed')['price']
	.agg(['count', 'mean', 'median', 'std'])
	.sort_values('median', ascending=False)
	.round(2)
)
print(barrio_stats)

barrio_max = barrio_stats['median'].idxmax()
barrio_min = barrio_stats['median'].idxmin()
ratio_barrios = barrio_stats.loc[barrio_max, 'median'] / barrio_stats.loc[barrio_min, 'median']
print(f"\n  Barrio más caro:   {barrio_max} (mediana €{barrio_stats.loc[barrio_max, 'median']:.0f})")
print(f"  Barrio más barato: {barrio_min} (mediana €{barrio_stats.loc[barrio_min, 'median']:.0f})")
print(f"  Ratio caro/barato: {ratio_barrios:.1f}x")

global_median = df_feat['price'].median()

# ══════════════════════════════════════════════════════════════════════
# FIGURAS
# ══════════════════════════════════════════════════════════════════════
plot_price_by_room_type(
	df_feat, rt_stats, save_path=viz_path / 'p2_boxplot_precio_room_type.png'
)

plot_price_by_neighbourhood(
	df_top, barrio_stats, global_median, top_n=top_n,
	save_path=viz_path / 'p2_barras_precio_barrio.png'
)

pivot_median = (
	df_top.pivot_table(values='price', index='neighbourhood_cleansed',
					   columns='room_type', aggfunc='median')
	.reindex(barrio_stats.index)
	.round(0)
)

plot_heatmap_neighbourhood_room_type(
	pivot_median, save_path=viz_path / 'p2_heatmap_barrio_room_type.png'
)

plot_price_per_person(
	df_feat, rt_stats, save_path=viz_path / 'p2_boxplot_price_per_person.png'
)

# ══════════════════════════════════════════════════════════════════════
# TABLA RESUMEN
# ══════════════════════════════════════════════════════════════════════
print(f"\n── Tabla resumen cruzada (medianas) ──")
print(pivot_median.to_string())

# ══════════════════════════════════════════════════════════════════════
# INSIGHT FINAL
# ══════════════════════════════════════════════════════════════════════
rt_max = rt_stats.index[0]
rt_max_med = rt_stats.iloc[0]['median']
rt_min = rt_stats.index[-1]
rt_min_med = rt_stats.iloc[-1]['median']

pp_entire = df_feat[df_feat['room_type'] == 'Entire home/apt']['price_per_person'].median() if 'price_per_person' in df_feat.columns else 0
pp_private = df_feat[df_feat['room_type'] == 'Private room']['price_per_person'].median() if 'price_per_person' in df_feat.columns else 0

print(f"\n{'─'*80}")
print(f"📊 INSIGHT — Pregunta 2: Diferencias de precio por tipo y barrio")
print(f"{'─'*80}")
print(f"  POR TIPO DE ALOJAMIENTO:")
print(f"  • {rt_max}: mediana €{rt_max_med:.0f}/noche (el más caro)")
print(f"  • {rt_min}: mediana €{rt_min_med:.0f}/noche (el más barato)")
print(f"  • Ratio: {rt_max_med / rt_min_med:.1f}x de diferencia")
if pp_entire > 0:
	print(f"  • Pero por persona: Entire home €{pp_entire:.0f}/pers vs Private room €{pp_private:.0f}/pers")
	print(f"    → Entire home es más EFICIENTE por persona para grupos")
print(f"")
print(f"  POR BARRIO (Top {top_n}):")
print(f"  • Barrio más caro:   {barrio_max} (mediana €{barrio_stats.loc[barrio_max, 'median']:.0f})")
print(f"  • Barrio más barato: {barrio_min} (mediana €{barrio_stats.loc[barrio_min, 'median']:.0f})")
print(f"  • Ratio geográfico:  {ratio_barrios:.1f}x de diferencia entre barrios")
print(f"  • La mediana global es €{global_median:.0f}/noche")
print(f"")
print(f"  CRUCE BARRIO × TIPO:")
print(f"  • Entire home en barrio premium puede costar {ratio_barrios:.0f}x más que")
print(f"    Private room en barrio económico")
print(f"  → La COMBINACIÓN ubicación + tipo genera la mayor dispersión de precios")
print(f"")
print(f"  CONCLUSIÓN:")
print(f"  → Tipo de alojamiento y barrio son los DOS principales segmentadores de precio")
print(f"  → Para viajeros: Private room en barrio periférico = opción más económica")
print(f"  → Para hosts: Entire home en zona centro = máxima tarifa posible")
print(f"{'─'*80}")

## 13) Pregunta 3 — ¿Ser Superhost se traduce en mayor precio o mayor demanda?
**Objetivo:** evaluar si el estatus de Superhost impacta en precio, volumen de reseñas y valoraciones.  
**Qué se hace:**
- Comparación de métricas clave entre Superhosts y No-Superhosts.
- Visualizaciones de distribución y resumen.

**Lectura esperada:** el badge de Superhost puede no implicar precios más altos, pero sí mayor actividad.

In [ ]:
print(f"\n{'='*80}")
print(f"PREGUNTA 3: ¿Ser Superhost se traduce en mayor precio o mayor demanda?")
print(f"{'='*80}")

viz_path = root / 'visualizations'

# ══════════════════════════════════════════════════════════════════════
# PARTE A: Estadísticas comparadas
# ══════════════════════════════════════════════════════════════════════
compare_cols = ['price', 'number_of_reviews', 'review_scores_rating', 'availability_365']
compare_cols = [c for c in compare_cols if c in df_feat.columns]

print(f"\n── Comparación Superhost vs No-Superhost ──")
metrics = {}
for col in compare_cols:
	mean_sh = df_feat[df_feat['host_is_superhost'] == True][col].mean()
	mean_no = df_feat[df_feat['host_is_superhost'] == False][col].mean()
	diff_pct = ((mean_sh - mean_no) / mean_no * 100) if mean_no != 0 else 0
	metrics[col] = diff_pct
	signo = "+" if diff_pct > 0 else ""
	print(f"  {col:<25s}  Superhost: {mean_sh:>8.1f}  |  No-SH: {mean_no:>8.1f}  |  Δ {signo}{diff_pct:.1f}%")

n_super = df_feat['host_is_superhost'].sum()
n_total = len(df_feat)
print(f"\n  Superhosts: {n_super:,} de {n_total:,} ({n_super/n_total*100:.1f}%)")

# ══════════════════════════════════════════════════════════════════════
# FIGURAS
# ══════════════════════════════════════════════════════════════════════
plot_superhost_boxplots(
	df_feat, save_path=viz_path / 'p3_boxplot_superhost.png'
)

plot_superhost_diff_bars(
	metrics, save_path=viz_path / 'p3_barras_superhost_diff.png'
)

# ══════════════════════════════════════════════════════════════════════
# INSIGHT FINAL
# ══════════════════════════════════════════════════════════════════════
price_sh = df_feat[df_feat['host_is_superhost'] == True]['price'].median()
price_no = df_feat[df_feat['host_is_superhost'] == False]['price'].median()
rev_sh = df_feat[df_feat['host_is_superhost'] == True]['number_of_reviews'].median()
rev_no = df_feat[df_feat['host_is_superhost'] == False]['number_of_reviews'].median()
rat_sh = df_feat[df_feat['host_is_superhost'] == True]['review_scores_rating'].median()
rat_no = df_feat[df_feat['host_is_superhost'] == False]['review_scores_rating'].median()

print(f"\n{'─'*80}")
print(f"📊 INSIGHT — Pregunta 3: Impacto de ser Superhost")
print(f"{'─'*80}")
print(f"")
print(f"  1. PRECIO: Superhosts NO cobran más")
print(f"     → Mediana Superhost: €{price_sh:.0f} vs No-SH: €{price_no:.0f}")
print(f"     → El badge NO se monetiza como premium de tarifa.")
print(f"")
print(f"  2. DEMANDA: Superhosts reciben MÁS reseñas")
print(f"     → Mediana reseñas SH: {rev_sh:.0f} vs No-SH: {rev_no:.0f}")
print(f"     → Mayor volumen = más reservas = más ingresos totales.")
print(f"")
print(f"  3. CALIDAD: Superhosts tienen MEJOR rating")
print(f"     → Mediana rating SH: {rat_sh:.2f} vs No-SH: {rat_no:.2f}")
print(f"     → Consistente con los requisitos del programa.")
print(f"")
print(f"  CONCLUSIÓN:")
print(f"  → Ser Superhost NO significa cobrar más, sino VENDER MÁS.")
print(f"  → La estrategia Superhost es de VOLUMEN, no de premium.")
print(f"  → Para hosts: el camino a más ingresos pasa por calidad de")
print(f"    servicio (→ Superhost) más que por subir tarifas.")
print(f"{'─'*80}")